# Finantal-LM — Supervised Fine-Tuning (SFT)

Loads the pretrained checkpoint from Drive (`init_from` defaults to `checkpoints/pretrain/latest.pt`) and fine-tunes on the SFT data. Auto-resumes from `checkpoints/sft/latest.pt` if present.


In [ ]:
# Check the GPU (expect a Tesla T4)
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# Mount Google Drive (data + tokenizer + checkpoints live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the code repo from GitHub
import os
REPO_URL = 'https://github.com/<YOUR_USERNAME>/finantal-lm.git'   # <-- EDIT
REPO_DIR = '/content/finantal-lm'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
%cd $REPO_DIR

In [ ]:
# Install dependencies (torch already on Colab — not reinstalled)
!pip -q install -r requirements.txt

In [ ]:
# Point the code at the data on Drive. The default already matches this path,
# but we set it explicitly so it's obvious and overridable.
import os
os.environ['FINANTAL_DATA_ROOT'] = '/content/drive/MyDrive/finantal_data'
# Make the repo importable from notebook cells too
import sys; sys.path.insert(0, '/content/finantal-lm')
from config import paths as P
print(P.summary())

In [ ]:
# Confirm the pretrained checkpoint exists on Drive
from config import paths as P; import os
assert os.path.exists(P.PRETRAIN_LATEST), f'Run pretraining first: {P.PRETRAIN_LATEST} missing'
print('pretrained checkpoint OK:', P.PRETRAIN_LATEST)

## Train (SFT)


In [ ]:
!python -m training.sft_train --config config/train_config.json \
    --override num_epochs=3

### Low-memory variant


In [ ]:
# !python -m training.sft_train --override micro_batch_size=2 gradient_accumulation_steps=16

## Loss curve


In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open('logs/sft_metrics.jsonl')]
plt.figure(figsize=(8,4)); plt.plot([r['step'] for r in rows], [r['loss'] for r in rows])
plt.xlabel('step'); plt.ylabel('loss'); plt.grid(True); plt.title('SFT loss'); plt.show()

## Chat sanity check


In [ ]:
!python -m training.sample --ckpt sft --prompt "ما هي الميزانية؟" --max_new_tokens 120 --temperature 0.7